In [3]:
! pip install --upgrade kiwipiepy

import json
import random
import re
from kiwipiepy import Kiwi

# ✅ 1. Kiwi 초기화 및 사용자 복합어 사전 등록
kiwi = Kiwi()

# 사용자 사전 등록: 형태소 분석 시 복합어(띄어쓰기 포함 단어)가 분리되지 않도록 설정
user_words = [
    '개혁신당', '더불어민주당', '국민의힘', '최고득표율',
    '이 후보', '인공지능', '직권남용', '양자대결',
    '김 후보', '출구조사', '선거법', '더중플', '권영국',
    '사고현장', '양회성'
]
# 각 단어를 고유명사(NNP)로 등록
for word in user_words:
    kiwi.add_user_word(word, "NNP")

# ✅ 2. 불용어 정의: 분석에 의미 없는 일반 단어 제거
stopwords = set([
    "것", "수", "등", "더", "때", "자신", "이번", "이후", "위해", "관련",
    "의견", "최근", "대해", "내용", "경우", "부분", "정도",
    "현재", "상황", "사실", "또한", "통해", "가장", "대한", "중요", "전혀",
    "하나", "바로", "계속", "여러", "많은", "이미",  "그", "저", "우리", "그것", "이것"
])

# ✅ 3. 제외할 품사 태그 정의: 조사, 어미, 숫자, 기호 등 분석에 불필요한 품사 제거
stop_tags = {
    'JKS', 'JKC', 'JKG', 'JKB', 'JKO', 'JKV', 'JKQ', 'JX', 'JC',  # 조사
    'NNB', 'NR', 'SN',                                           # 의존명사, 수사, 숫자
    'SF', 'SP', 'SS', 'SE', 'SO',                                # 문장부호, 기호
    'EP', 'EF', 'EC', 'ETN', 'ETM',                              # 어미
    'XPN', 'XSN', 'XSV', 'XSA',                                  # 접사
    'IC', 'VSV'                                                  # 감탄사, 기타
}




  Using cached kiwipiepy-0.21.0-cp312-cp312-win_amd64.whl.metadata (1.3 kB)
  Using cached kiwipiepy_model-0.21.0-py3-none-any.whl
Using cached kiwipiepy-0.21.0-cp312-cp312-win_amd64.whl (2.4 MB)


ERROR: Could not install packages due to an OSError: [WinError 32] 다른 프로세스가 파일을 사용 중이기 때문에 프로세스가 액세스 할 수 없습니다: 'C:\\Users\\gram\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\site-packages\\kiwipiepy_model\\sj.knlm'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# ✅ 4. JSON 기사 파일을 읽어와 통합하는 함수
def load_articles(*filepaths):
    articles = []
    for path in filepaths:
        with open(path, encoding='utf-8') as f:
            articles.extend(json.load(f))
    return articles

In [ ]:
# ✅ 5. 지정된 키워드들이 함께 등장하는 문장을 추출하는 함수
def extract_sentences_with_keywords(articles, keywords, stopwords, stop_tags, sample_n=10):
    matched = []

    for article in articles:
        content = article.get("content", "")
        article_id = article.get("id", "UNKNOWN_ID")

        # 문장을 마침표/물음표/느낌표/어미 기준으로 분리
        sentence_split_pattern = r'(?<=[.?!다요])\s+'
        sentences = re.split(sentence_split_pattern, content)


        for sentence in sentences:
            # 형태소 분석: 불필요한 품사와 불용어를 제거한 단어 리스트 생성
            tokens = kiwi.tokenize(sentence)
            words = [
                tok.form for tok in tokens
                if tok.tag not in stop_tags and tok.form not in stopwords
            ]

            # 검색 키워드 중 몇 개가 문장에 등장하는지 확인
            overlap = set(words) & set(keywords)
            if all(term in words for term in search_terms):
                matched.append({
                    "sentence": sentence.strip(),         # 문장 내용
                    "article_id": article_id,             # 해당 문장이 포함된 기사 ID
                    "matched_words": list(overlap)        # 함께 등장한 키워드 리스트
                })

    # 조건에 맞는 문장에서 최대 sample_n개 랜덤 추출
    return random.sample(matched, min(sample_n, len(matched)))

In [6]:
# ✅ 6. 기사 불러오기 (필요한 JSON 파일 지정)
articles = load_articles("이재명.json", "김문수.json", "이준석.json")

In [ ]:
# 실행 예시

'''
# ✅ 7. 찾고 싶은 단어 지정 (이 리스트 안의 단어들 중 2개 이상 함께 등장하는 문장을 추출)
search_terms = ["ㅇㅇㅇㅇㅇ", "ㅁㅁㅁㅁㅁ", "ㅁㅇㅁㅇㅁㅇ"]  # ← 이 부분만 바꿔가며 사용하면 됩니다

# ✅ 8. 문장 추출 실행
samples = extract_sentences_with_keywords(
    articles=articles,
    keywords=search_terms,
    stopwords=stopwords,
    stop_tags=stop_tags,
    sample_n=10  # 랜덤 추출 개수
)

# ✅ 9. 결과 출력
for idx, item in enumerate(samples, 1):
    print(f"[{idx}] {item['sentence']}")
    print(f"     📄 기사 ID: {item['article_id']}")
    print(f"     🔍 함께 등장한 단어: {item['matched_words']}\n")

'''

In [ ]:
search_terms = ["김문수", "단일"]  # ← 이 부분만 바꿔가며 사용하면 됩니다

# ✅ 문장 추출 실행
samples = extract_sentences_with_keywords(
    articles=articles,
    keywords=search_terms,
    stopwords=stopwords,
    stop_tags=stop_tags,
    sample_n=5  # 랜덤 추출 개수
)

# ✅결과 출력
for idx, item in enumerate(samples, 1):
    print(f"[{idx}] {item['sentence']}")
    print(f"     📄 기사 ID: {item['article_id']}")
    print(f"     🔍 함께 등장한 단어: {item['matched_words']}\n")

In [ ]:
search_terms = ["이준석", "단일"]  # ← 이 부분만 바꿔가며 사용하면 됩니다

# ✅ 문장 추출 실행
samples = extract_sentences_with_keywords(
    articles=articles,
    keywords=search_terms,
    stopwords=stopwords,
    stop_tags=stop_tags,
    sample_n=5  # 랜덤 추출 개수
)

# ✅결과 출력
for idx, item in enumerate(samples, 1):
    print(f"[{idx}] {item['sentence']}")
    print(f"     📄 기사 ID: {item['article_id']}")
    print(f"     🔍 함께 등장한 단어: {item['matched_words']}\n")

In [ ]:
search_terms = ["이재명", "법원"]  # ← 이 부분만 바꿔가며 사용하면 됩니다

# ✅ 문장 추출 실행
samples = extract_sentences_with_keywords(
    articles=articles,
    keywords=search_terms,
    stopwords=stopwords,
    stop_tags=stop_tags,
    sample_n=5  # 랜덤 추출 개수
)

# ✅결과 출력
for idx, item in enumerate(samples, 1):
    print(f"[{idx}] {item['sentence']}")
    print(f"     📄 기사 ID: {item['article_id']}")
    print(f"     🔍 함께 등장한 단어: {item['matched_words']}\n")

In [ ]:
search_terms = ["이재명", "사건"]  # ← 이 부분만 바꿔가며 사용하면 됩니다

# ✅ 문장 추출 실행
samples = extract_sentences_with_keywords(
    articles=articles,
    keywords=search_terms,
    stopwords=stopwords,
    stop_tags=stop_tags,
    sample_n=5  # 랜덤 추출 개수
)

# ✅결과 출력
for idx, item in enumerate(samples, 1):
    print(f"[{idx}] {item['sentence']}")
    print(f"     📄 기사 ID: {item['article_id']}")
    print(f"     🔍 함께 등장한 단어: {item['matched_words']}\n")

In [ ]:
search_terms = ["김문수", "대표"]  # ← 이 부분만 바꿔가며 사용하면 됩니다

# ✅ 문장 추출 실행
samples = extract_sentences_with_keywords(
    articles=articles,
    keywords=search_terms,
    stopwords=stopwords,
    stop_tags=stop_tags,
    sample_n=5  # 랜덤 추출 개수
)

# ✅결과 출력
for idx, item in enumerate(samples, 1):
    print(f"[{idx}] {item['sentence']}")
    print(f"     📄 기사 ID: {item['article_id']}")
    print(f"     🔍 함께 등장한 단어: {item['matched_words']}\n")

In [ ]:
search_terms = ["이준석", "토론"]  # ← 이 부분만 바꿔가며 사용하면 됩니다

# ✅ 문장 추출 실행
samples = extract_sentences_with_keywords(
    articles=articles,
    keywords=search_terms,
    stopwords=stopwords,
    stop_tags=stop_tags,
    sample_n=5  # 랜덤 추출 개수
)

# ✅결과 출력
for idx, item in enumerate(samples, 1):
    print(f"[{idx}] {item['sentence']}")
    print(f"     📄 기사 ID: {item['article_id']}")
    print(f"     🔍 함께 등장한 단어: {item['matched_words']}\n")

In [ ]:
search_terms = ["이준석", "공약"]  # ← 이 부분만 바꿔가며 사용하면 됩니다

# ✅ 문장 추출 실행
samples = extract_sentences_with_keywords(
    articles=articles,
    keywords=search_terms,
    stopwords=stopwords,
    stop_tags=stop_tags,
    sample_n=5  # 랜덤 추출 개수
)

# ✅결과 출력
for idx, item in enumerate(samples, 1):
    print(f"[{idx}] {item['sentence']}")
    print(f"     📄 기사 ID: {item['article_id']}")
    print(f"     🔍 함께 등장한 단어: {item['matched_words']}\n")

In [ ]:
search_terms = ["이재명", "통합"]  # ← 이 부분만 바꿔가며 사용하면 됩니다

# ✅ 문장 추출 실행
samples = extract_sentences_with_keywords(
    articles=articles,
    keywords=search_terms,
    stopwords=stopwords,
    stop_tags=stop_tags,
    sample_n=5  # 랜덤 추출 개수
)

# ✅결과 출력
for idx, item in enumerate(samples, 1):
    print(f"[{idx}] {item['sentence']}")
    print(f"     📄 기사 ID: {item['article_id']}")
    print(f"     🔍 함께 등장한 단어: {item['matched_words']}\n")

In [ ]:
search_terms = ["김문수", "윤석열"]  # ← 이 부분만 바꿔가며 사용하면 됩니다

# ✅ 문장 추출 실행
samples = extract_sentences_with_keywords(
    articles=articles,
    keywords=search_terms,
    stopwords=stopwords,
    stop_tags=stop_tags,
    sample_n=5  # 랜덤 추출 개수
)

# ✅결과 출력
for idx, item in enumerate(samples, 1):
    print(f"[{idx}] {item['sentence']}")
    print(f"     📄 기사 ID: {item['article_id']}")
    print(f"     🔍 함께 등장한 단어: {item['matched_words']}\n")